# AI-Based Motorcycle Helmet Detection - YOLOv8 Training

This notebook trains YOLOv8 object detection models for Theme 4: motorcycle helmet detection.

For team training, assign one model size and one dataset to each trainer:

- Person 1: `ASSIGNED_YOLO_SIZE = 'n'`, `ASSIGNED_DATASET_KEYS = ['data']`
- Person 2: `ASSIGNED_YOLO_SIZE = 'n'`, `ASSIGNED_DATASET_KEYS = ['data_combined']`
- Person 3: `ASSIGNED_YOLO_SIZE = 's'`, `ASSIGNED_DATASET_KEYS = ['data']`
- Person 4: `ASSIGNED_YOLO_SIZE = 's'`, `ASSIGNED_DATASET_KEYS = ['data_combined']`

Each trainer will automatically run two experiments:

- assigned model + assigned dataset + 60 epochs
- assigned model + assigned dataset + 100 epochs

Expected Drive folder:

`/content/drive/MyDrive/Helmet-Detection`

The notebook currently creates only `train` and `valid` splits. A separate held-out test set can be added later after collection.

Each run saves to a separate folder such as `training_runs/helmet_data_yolov8n_e60` or `training_runs/helmet_data_combined_yolov8s_e100`, then exports a separate model such as `exported_models/helmet_data_combined_yolov8s_e100_best.pt`.

## 1. Install YOLOv8 and Mount Google Drive

In [ ]:
# Code Cell 1
%pip install -q ultralytics pyyaml pandas matplotlib opencv-python

from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Code Cell 2
from pathlib import Path
import random
import shutil
import yaml
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import cv2
import torch
from ultralytics import YOLO

# Main project directory in Google Drive
PROJECT_DIR = Path('/content/drive/MyDrive/Helmet-Detection')

# Change only these two settings for each trainer.
# Model size options: 'n' or 's'
ASSIGNED_YOLO_SIZE = ''

# Dataset options: ['data'] or ['data_combined']
# Use ['data', 'data_combined'] only if one person intentionally wants to run both datasets.
ASSIGNED_DATASET_KEYS = ['']

assert ASSIGNED_YOLO_SIZE in {'n', 's'}, "ASSIGNED_YOLO_SIZE must be 'n' or 's'"
assert ASSIGNED_DATASET_KEYS, 'ASSIGNED_DATASET_KEYS must contain at least one dataset key.'

YOLO_SIZES = [ASSIGNED_YOLO_SIZE]
EPOCH_OPTIONS = [60, 100]

# Dataset aliases become part of each run name.
ALL_DATASETS = {
    'data': PROJECT_DIR / 'data',
    'data_combined': PROJECT_DIR / 'data-combined',
}
unknown_dataset_keys = set(ASSIGNED_DATASET_KEYS) - set(ALL_DATASETS)
assert not unknown_dataset_keys, f'Unknown dataset keys: {sorted(unknown_dataset_keys)}'
DATASETS = {dataset_key: ALL_DATASETS[dataset_key] for dataset_key in ASSIGNED_DATASET_KEYS}

# If a dataset only has a train split, this notebook creates deterministic
# train/valid working splits under /content without changing Drive files.
SPLIT_SEED = 42
VALID_FRACTION = 0.20
WORKING_DATA_DIR = Path('/content/helmet_detection_working')

# Absolute YAML files generated for Colab training.
COLAB_YAML_DIR = Path('/content/helmet_detection_yamls')
COLAB_YAML_DIR.mkdir(parents=True, exist_ok=True)

# Copy checkpoints out of the Drive mount before validation/prediction.
STAGED_MODEL_DIR = Path('/content/staged_models')
STAGED_MODEL_DIR.mkdir(parents=True, exist_ok=True)


def stage_model_for_inference(model_path, run_name=None):
    model_path = Path(model_path)
    assert model_path.exists(), f'Model file not found: {model_path}'

    staged_name = f'{run_name or model_path.stem}{model_path.suffix}'
    staged_path = STAGED_MODEL_DIR / staged_name

    source_size = model_path.stat().st_size
    if source_size <= 0:
        raise OSError(f'Model file is empty: {model_path}')

    if staged_path.exists() and staged_path.stat().st_size == source_size:
        print('Using staged model copy:', staged_path)
        return staged_path

    shutil.copy2(model_path, staged_path)
    copied_size = staged_path.stat().st_size
    if copied_size != source_size:
        raise OSError(
            f'Copied model size mismatch for {model_path}: source={source_size}, staged={copied_size}'
        )

    print('Staged model copy:', staged_path)
    return staged_path


# Move output outside of dataset folders.
OUTPUT_DIR = PROJECT_DIR / 'training_runs'
EXPORT_DIR = PROJECT_DIR / 'exported_models'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

for dataset_key, dataset_dir in DATASETS.items():
    source_yaml = dataset_dir / 'data.yaml'
    assert dataset_dir.exists(), f'Dataset folder not found: {dataset_dir}'
    assert source_yaml.exists(), f'data.yaml not found: {source_yaml}'

print('Project directory:', PROJECT_DIR)
print('Assigned YOLO size:', ASSIGNED_YOLO_SIZE)
print('Assigned dataset keys:', ASSIGNED_DATASET_KEYS)
print('Epoch options:', EPOCH_OPTIONS)
print('Datasets:')
for dataset_key, dataset_dir in DATASETS.items():
    print(f'  {dataset_key}: {dataset_dir}')
print('Working data directory:', WORKING_DATA_DIR)
print('Output directory:', OUTPUT_DIR)
print('Export directory:', EXPORT_DIR)
print('Staged model directory:', STAGED_MODEL_DIR)
print('GPU available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))


## 2. Validate Dataset Layout

This confirms image/label counts for the assigned source dataset. If the dataset has only `train`, the next cell creates deterministic working `train` and `valid` folders under `/content` before writing an absolute Colab YAML file.

In [ ]:
# Code Cell 3
data_configs = {}
rows = []

for dataset_key, dataset_dir in DATASETS.items():
    source_yaml = dataset_dir / 'data.yaml'
    with open(source_yaml, 'r') as f:
        data_config = yaml.safe_load(f)
    data_configs[dataset_key] = data_config

    for split in ['train', 'valid']:
        image_dir = dataset_dir / split / 'images'
        label_dir = dataset_dir / split / 'labels'
        rows.append({
            'dataset': dataset_key,
            'split': split,
            'images': len(list(image_dir.glob('*'))) if image_dir.exists() else 0,
            'labels': len(list(label_dir.glob('*.txt'))) if label_dir.exists() else 0,
            'image_dir_exists': image_dir.exists(),
            'label_dir_exists': label_dir.exists(),
        })

dataset_table = pd.DataFrame(rows)
display(dataset_table)

train_rows = dataset_table[dataset_table['split'] == 'train'].set_index('dataset')
missing_train_images = train_rows[~train_rows['image_dir_exists'] | (train_rows['images'] == 0)].index.tolist()
missing_train_labels = train_rows[~train_rows['label_dir_exists']].index.tolist()
assert not missing_train_images, f'No training images found for: {missing_train_images}'
assert not missing_train_labels, f'Training label folder missing for: {missing_train_labels}'

class_names = {dataset_key: tuple(config.get('names', [])) for dataset_key, config in data_configs.items()}
assert len(set(class_names.values())) == 1, f'Datasets have different class names: {class_names}'
print('Class names:', next(iter(class_names.values())))

In [ ]:
# Code Cell 4
IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
WORKING_DATASETS = {}
COLAB_YAMLS = {}


def image_files(image_dir):
    return sorted(path for path in image_dir.glob('*') if path.suffix.lower() in IMAGE_EXTENSIONS)


def matching_label(image_path, label_dir):
    return label_dir / f'{image_path.stem}.txt'


def link_or_copy(src, dst):
    dst.parent.mkdir(parents=True, exist_ok=True)
    if dst.exists() or dst.is_symlink():
        dst.unlink()
    try:
        dst.symlink_to(src)
    except OSError:
        shutil.copy2(src, dst)


def write_split(files, source_label_dir, target_dir, split):
    image_target = target_dir / split / 'images'
    label_target = target_dir / split / 'labels'
    image_target.mkdir(parents=True, exist_ok=True)
    label_target.mkdir(parents=True, exist_ok=True)

    for image_path in files:
        link_or_copy(image_path, image_target / image_path.name)
        label_path = matching_label(image_path, source_label_dir)
        target_label = label_target / f'{image_path.stem}.txt'
        if label_path.exists():
            link_or_copy(label_path, target_label)
        else:
            target_label.touch()


for dataset_key, dataset_dir in DATASETS.items():
    target_dir = WORKING_DATA_DIR / dataset_key
    if target_dir.exists():
        shutil.rmtree(target_dir)

    train_images = image_files(dataset_dir / 'train' / 'images')
    train_labels_dir = dataset_dir / 'train' / 'labels'
    valid_images = image_files(dataset_dir / 'valid' / 'images')

    if valid_images:
        split_files = {
            'train': train_images,
            'valid': valid_images,
        }
        split_label_dirs = {
            'train': dataset_dir / 'train' / 'labels',
            'valid': dataset_dir / 'valid' / 'labels',
        }
    else:
        shuffled = train_images.copy()
        random.Random(SPLIT_SEED).shuffle(shuffled)
        total = len(shuffled)
        valid_count = max(1, round(total * VALID_FRACTION)) if total >= 2 else 0
        train_count = total - valid_count

        split_files = {
            'train': shuffled[:train_count],
            'valid': shuffled[train_count:],
        }
        split_label_dirs = {split: train_labels_dir for split in ['train', 'valid']}

    for split, files in split_files.items():
        write_split(files, split_label_dirs[split], target_dir, split)

    WORKING_DATASETS[dataset_key] = target_dir

    data_config = data_configs[dataset_key]
    colab_config = {
        'train': str(target_dir / 'train' / 'images'),
        'val': str(target_dir / 'valid' / 'images'),
        'nc': int(data_config.get('nc', 3)),
        'names': data_config.get('names', ['helmet', 'human', 'motorcycle']),
    }

    colab_yaml = COLAB_YAML_DIR / f'{dataset_key}.yaml'
    with open(colab_yaml, 'w') as f:
        yaml.safe_dump(colab_config, f, sort_keys=False)
    COLAB_YAMLS[dataset_key] = colab_yaml

    split_counts = {split: len(files) for split, files in split_files.items()}
    print(f'--- {dataset_key} working split counts: {split_counts} ---')
    print(colab_yaml.read_text())

## 3. Preview Training Images

In [ ]:
# Code Cell 5
for dataset_key, dataset_dir in WORKING_DATASETS.items():
    sample_images = sorted((dataset_dir / 'train' / 'images').glob('*'))[:6]
    fig, axes = plt.subplots(2, 3, figsize=(14, 8))
    axes = axes.flatten()
    fig.suptitle(f'Training samples: {dataset_key}', fontsize=14)

    for ax, image_path in zip(axes, sample_images):
        image = cv2.cvtColor(cv2.imread(str(image_path)), cv2.COLOR_BGR2RGB)
        ax.imshow(image)
        ax.set_title(image_path.name[:40])
        ax.axis('off')

    for ax in axes[len(sample_images):]:
        ax.axis('off')

    plt.tight_layout()
    plt.show()

## 4. Train YOLOv8

This cell trains the assigned model size on the assigned dataset for both epoch settings. With the default assignment, it runs:

- `helmet_data_yolov8n_e60`
- `helmet_data_yolov8n_e100`

For the other trainers, change `ASSIGNED_YOLO_SIZE` and `ASSIGNED_DATASET_KEYS` in section 1 before running the notebook.

Batch size is chosen automatically by model size to reduce Colab out-of-memory errors. If a run crashes from memory, reduce that model size's batch value below.

In [ ]:
# Code Cell 6
IMAGE_SIZE = 640
BATCH_SIZE_BY_SIZE = {
    'n': 64,
    's': 16,
}
DEVICE = 0 if torch.cuda.is_available() else 'cpu'
SKIP_FINISHED_RUNS = True

RUNS = []

for yolo_size in YOLO_SIZES:
    for epochs in EPOCH_OPTIONS:
        for dataset_key, dataset_dir in WORKING_DATASETS.items():
            model_variant = f'yolov8{yolo_size}.pt'
            run_name = f'helmet_{dataset_key}_yolov8{yolo_size}_e{epochs}'
            batch_size = BATCH_SIZE_BY_SIZE[yolo_size]
            colab_yaml = COLAB_YAMLS[dataset_key]
            best_model = OUTPUT_DIR / run_name / 'weights' / 'best.pt'
            last_model = OUTPUT_DIR / run_name / 'weights' / 'last.pt'

            print('=' * 80)
            print(f'Training {model_variant} on {dataset_key} for {epochs} epochs as {run_name}')
            print('Image size:', IMAGE_SIZE)
            print('Batch size:', batch_size)
            print('Data YAML:', colab_yaml)

            if SKIP_FINISHED_RUNS and best_model.exists():
                print('Skipping training because best.pt already exists:', best_model)
            else:
                model = YOLO(model_variant)
                model.train(
                    data=str(colab_yaml),
                    epochs=epochs,
                    imgsz=IMAGE_SIZE,
                    batch=batch_size,
                    patience=0,
                    project=str(OUTPUT_DIR),
                    name=run_name,
                    exist_ok=True,
                    seed=42,
                    device=DEVICE,
                    optimizer='auto',
                    cos_lr=True,
                    plots=True,
                    cache="disk",
                )

                del model
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()

            print('Best model:', best_model)
            print('Last model:', last_model)

            RUNS.append({
                'dataset': dataset_key,
                'dataset_dir': dataset_dir,
                'yolo_size': yolo_size,
                'epochs': epochs,
                'model_variant': model_variant,
                'run_name': run_name,
                'batch_size': batch_size,
                'colab_yaml': colab_yaml,
                'best_model': best_model,
                'last_model': last_model,
            })

run_table = pd.DataFrame([{k: str(v) if isinstance(v, Path) else v for k, v in run.items()} for run in RUNS])
display(run_table)

## 5. Validate on the Validation Split

In [ ]:
# Code Cell 7
comparison_rows = []

for run in RUNS:
    print('=' * 80)
    print(f"Validating {run['run_name']}")

    staged_model_path = stage_model_for_inference(run['best_model'], run['run_name'])
    best_model = YOLO(str(staged_model_path))
    # conf is intentionally omitted so YOLO uses its default (0.001),
    # which sweeps the full precision-recall curve for a standard mAP score.
    validation_metrics = best_model.val(
        data=str(run['colab_yaml']),
        split='val',
        imgsz=IMAGE_SIZE,
        device=DEVICE,
    )

    # Per-class AP50 — index matches data.yaml 'names' order.
    class_names_list = list(data_configs[run['dataset']].get('names', []))
    per_class_ap50 = validation_metrics.box.ap50  # numpy array, one value per class
    per_class_metrics = {
        f'ap50_{name}': float(per_class_ap50[i])
        for i, name in enumerate(class_names_list)
        if i < len(per_class_ap50)
    }

    row = {
        'dataset': run['dataset'],
        'run_name': run['run_name'],
        'model_variant': run['model_variant'],
        'yolo_size': run['yolo_size'],
        'epochs': run['epochs'],
        'image_size': IMAGE_SIZE,
        'batch_size': run['batch_size'],
        'val_model_path': str(run['best_model']),
        'val_staged_model_path': str(staged_model_path),
        'val_map50_95': float(validation_metrics.box.map),
        'val_map50': float(validation_metrics.box.map50),
        'val_precision': float(validation_metrics.box.mp),
        'val_recall': float(validation_metrics.box.mr),
    }
    row.update(per_class_metrics)
    comparison_rows.append(row)

    del best_model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

new_comparison_df = pd.DataFrame(comparison_rows)
comparison_path = OUTPUT_DIR / 'model_comparison.csv'

if comparison_path.exists():
    comparison_df = pd.read_csv(comparison_path)
    comparison_df = comparison_df[~comparison_df['run_name'].isin(new_comparison_df['run_name'])]
    comparison_df = pd.concat([comparison_df, new_comparison_df], ignore_index=True)
else:
    comparison_df = new_comparison_df

comparison_df = comparison_df.sort_values(['yolo_size', 'epochs', 'dataset']).reset_index(drop=True)
comparison_df.to_csv(comparison_path, index=False)
display(comparison_df)
print('Comparison CSV:', comparison_path)


## 6. External Test Set Later

Skip test predictions for now. After the team collects a true held-out test set, add its image folder here and run prediction/evaluation only once on the final candidate models.

If you open a fresh Colab session later, rerun section 1, rerun the setup/import cell that defines `PROJECT_DIR`, then run the helper cell below to rebuild the saved model list from Google Drive before running the test cell.

This section is set up for the full team. If all four people save their final `.pt` files into the shared Drive project, the helper will automatically collect every saved run, including:

- Person 1: `ASSIGNED_YOLO_SIZE = 'n'`, `ASSIGNED_DATASET_KEYS = ['data']`
- Person 2: `ASSIGNED_YOLO_SIZE = 'n'`, `ASSIGNED_DATASET_KEYS = ['data_combined']`
- Person 3: `ASSIGNED_YOLO_SIZE = 's'`, `ASSIGNED_DATASET_KEYS = ['data']`
- Person 4: `ASSIGNED_YOLO_SIZE = 's'`, `ASSIGNED_DATASET_KEYS = ['data_combined']`

When labels are present for the external test set, this section will save report-ready CSV summaries, per-class metrics, plots, and qualitative prediction outputs.

In [ ]:
# Code Cell 8
# Fresh-session helper for section 6.
# In a new Colab session, rerun section 1 and the setup/import cell first,
# then run this cell to rebuild the saved model list from Drive.

import re

EXTERNAL_TEST_MODEL_NAMES = []
# Example manual override: ['helmet_data_combined_yolov8s_e60', 'helmet_data_combined_yolov8s_e100']
# Leave empty to auto-select models from ASSIGNED_YOLO_SIZE and ASSIGNED_DATASET_KEYS.

TEAM_ASSIGNMENTS = [
    {'person': 'Person 1', 'yolo_size': 'n', 'dataset_keys': 'data'},
    {'person': 'Person 2', 'yolo_size': 'n', 'dataset_keys': 'data_combined'},
    {'person': 'Person 3', 'yolo_size': 's', 'dataset_keys': 'data'},
    {'person': 'Person 4', 'yolo_size': 's', 'dataset_keys': 'data_combined'},
]
display(pd.DataFrame(TEAM_ASSIGNMENTS))

RUN_NAME_PATTERN = re.compile(r'helmet_(?P<dataset>.+)_yolov8(?P<yolo_size>[ns])_e(?P<epochs>\d+)')


def parse_saved_run_metadata(run_name):
    match = RUN_NAME_PATTERN.fullmatch(run_name)
    if not match:
        return {'dataset': None, 'yolo_size': None, 'epochs': None}
    metadata = match.groupdict()
    metadata['epochs'] = int(metadata['epochs'])
    return metadata


def build_saved_run(model_path, exported_model):
    run_name = model_path.name.removesuffix('_best.pt') if exported_model else model_path.parent.parent.name
    metadata = parse_saved_run_metadata(run_name)
    saved_run = {
        'run_name': run_name,
        'best_model': model_path,
        'dataset': metadata['dataset'],
        'yolo_size': metadata['yolo_size'],
        'epochs': metadata['epochs'],
    }
    if metadata['yolo_size']:
        saved_run['model_variant'] = f"yolov8{metadata['yolo_size']}.pt"
    return saved_run


def normalize_class_names(raw_names):
    if isinstance(raw_names, dict):
        return [raw_names[key] for key in sorted(raw_names)]
    return list(raw_names)


TEST_RUNS = []
run_source_description = ''
selection_description = 'all saved models'

if 'RUNS' in globals() and RUNS:
    TEST_RUNS = RUNS
    run_source_description = 'current notebook session'
else:
    exported_model_paths = sorted(EXPORT_DIR.glob('*_best.pt'))
    if exported_model_paths:
        TEST_RUNS = [build_saved_run(model_path, exported_model=True) for model_path in exported_model_paths]
        run_source_description = f'exported models in {EXPORT_DIR}'
    else:
        training_best_paths = sorted(OUTPUT_DIR.glob('*/weights/best.pt'))
        TEST_RUNS = [build_saved_run(model_path, exported_model=False) for model_path in training_best_paths]
        run_source_description = f'training runs in {OUTPUT_DIR}'

if EXTERNAL_TEST_MODEL_NAMES:
    selected_run_names = set(EXTERNAL_TEST_MODEL_NAMES)
    TEST_RUNS = [run for run in TEST_RUNS if run['run_name'] in selected_run_names]
    assert TEST_RUNS, (
        'None of EXTERNAL_TEST_MODEL_NAMES matched a saved model. '
        f'Checked: {sorted(selected_run_names)}'
    )
    selection_description = f'manual model names: {sorted(selected_run_names)}'
else:
    assigned_yolo_size = globals().get('ASSIGNED_YOLO_SIZE')
    assigned_dataset_keys = list(globals().get('ASSIGNED_DATASET_KEYS', []))
    if assigned_yolo_size and assigned_dataset_keys:
        TEST_RUNS = [
            run for run in TEST_RUNS
            if run.get('yolo_size') == assigned_yolo_size and run.get('dataset') in assigned_dataset_keys
        ]
        assert TEST_RUNS, (
            'No saved models matched the assigned settings: '
            f'ASSIGNED_YOLO_SIZE={assigned_yolo_size}, '
            f'ASSIGNED_DATASET_KEYS={assigned_dataset_keys}'
        )
        selection_description = (
            f'assigned settings: size={assigned_yolo_size}, datasets={assigned_dataset_keys}'
        )

assert TEST_RUNS, (
    f'No saved models found in {EXPORT_DIR} or {OUTPUT_DIR}. '
    'Run section 7 once after training so later sessions can reload the saved weights.'
)

TEST_RUNS = sorted(
    TEST_RUNS,
    key=lambda run: (
        run.get('yolo_size') or '',
        run.get('dataset') or '',
        run.get('epochs') or 0,
        run['run_name'],
    ),
)

TEST_CLASS_NAMES = []
for dataset_yaml in [PROJECT_DIR / 'data' / 'data.yaml', PROJECT_DIR / 'data-combined' / 'data.yaml']:
    if dataset_yaml.exists():
        with open(dataset_yaml, 'r') as f:
            dataset_config = yaml.safe_load(f)
        TEST_CLASS_NAMES = normalize_class_names(dataset_config.get('names', []))
        if TEST_CLASS_NAMES:
            break

assert TEST_CLASS_NAMES, 'Could not load class names from data/data.yaml or data-combined/data.yaml.'

TEST_IMAGE_SIZE = globals().get('IMAGE_SIZE', 640)
TEST_DEVICE = globals().get('DEVICE', 0 if torch.cuda.is_available() else 'cpu')

print(f'Prepared {len(TEST_RUNS)} model(s) from {run_source_description}.')
print('Model selection:', selection_description)
print('External test image size:', TEST_IMAGE_SIZE)
print('External test device:', TEST_DEVICE)
print('External test classes:', TEST_CLASS_NAMES)
for run in TEST_RUNS:
    print(
        f"  - {run['run_name']} | dataset={run.get('dataset')} | "
        f"size={run.get('yolo_size')} | epochs={run.get('epochs')} | "
        f"model={run['best_model']}"
    )


In [ ]:
# Code Cell 9
# Set to True once a real held-out test set has been collected.
# Until then this cell is safe to run - it will print a reminder and exit early.
RUN_TEST_EVAL = False

TEST_IMAGE_DIR = PROJECT_DIR / 'test' / 'images'  # update path when ready
TEST_LABEL_DIR = PROJECT_DIR / 'test' / 'labels'
TEST_CONF = 0.25
TEST_RUNS_DIR = PROJECT_DIR / 'test_runs'
TEST_SAVE_PREDICTION_IMAGES = True
TEST_SAVE_PREDICTION_LABELS = True
TEST_SAVE_PREDICTION_CONF = True
TEST_SAVE_COMPARISON_IMAGES = True

if not RUN_TEST_EVAL:
    print('Test evaluation skipped. Set RUN_TEST_EVAL = True and supply TEST_IMAGE_DIR when a held-out test set is ready.')
else:
    assert 'TEST_RUNS' in globals() and TEST_RUNS, 'Run the helper cell above first to rebuild the saved model list.'
    assert TEST_IMAGE_DIR.exists(), f'Test image folder not found: {TEST_IMAGE_DIR}'

    test_image_extensions = globals().get('IMAGE_EXTENSIONS', {'.jpg', '.jpeg', '.png', '.bmp', '.webp'})
    test_image_paths = sorted(path for path in TEST_IMAGE_DIR.glob('*') if path.suffix.lower() in test_image_extensions)
    assert test_image_paths, f'No test images found in: {TEST_IMAGE_DIR}'

    test_label_paths = sorted(TEST_LABEL_DIR.glob('*.txt')) if TEST_LABEL_DIR.exists() else []
    TEST_HAS_LABELS = bool(test_label_paths)
    test_label_map = {path.stem: path for path in test_label_paths}

    TEST_RUNS_DIR.mkdir(parents=True, exist_ok=True)

    print('Test root directory:', TEST_RUNS_DIR)
    print('Test image folder:', TEST_IMAGE_DIR)
    print('Test label folder:', TEST_LABEL_DIR)
    print('Test images found:', len(test_image_paths))
    print('Test label files found:', len(test_label_paths))
    print('Label-aware metrics enabled:', TEST_HAS_LABELS)
    print('Save side-by-side comparisons:', TEST_SAVE_COMPARISON_IMAGES and TEST_HAS_LABELS)

    external_test_summary_rows = []
    external_test_per_class_rows = []
    external_test_image_rows = []

    def class_color(class_id):
        palette = [
            (0, 200, 0),
            (0, 140, 255),
            (255, 140, 0),
            (180, 0, 180),
            (0, 200, 200),
            (200, 200, 0),
        ]
        return palette[class_id % len(palette)]

    def yolo_label_to_xyxy(x_center, y_center, box_width, box_height, image_width, image_height):
        x1 = int(round((x_center - box_width / 2) * image_width))
        y1 = int(round((y_center - box_height / 2) * image_height))
        x2 = int(round((x_center + box_width / 2) * image_width))
        y2 = int(round((y_center + box_height / 2) * image_height))
        x1 = max(0, min(x1, image_width - 1))
        y1 = max(0, min(y1, image_height - 1))
        x2 = max(0, min(x2, image_width - 1))
        y2 = max(0, min(y2, image_height - 1))
        return x1, y1, x2, y2

    def draw_labeled_boxes(image_bgr, label_path, class_names):
        drawn = image_bgr.copy()
        if label_path is None or not label_path.exists():
            return drawn

        image_height, image_width = drawn.shape[:2]
        with open(label_path, 'r') as f:
            for raw_line in f:
                parts = raw_line.strip().split()
                if len(parts) < 5:
                    continue
                class_id = int(float(parts[0]))
                x_center, y_center, box_width, box_height = map(float, parts[1:5])
                x1, y1, x2, y2 = yolo_label_to_xyxy(
                    x_center, y_center, box_width, box_height, image_width, image_height
                )
                color = class_color(class_id)
                class_name = class_names[class_id] if 0 <= class_id < len(class_names) else f'class_{class_id}'
                cv2.rectangle(drawn, (x1, y1), (x2, y2), color, 2)
                text_y = y1 - 10 if y1 > 20 else y1 + 20
                cv2.putText(
                    drawn,
                    f'GT: {class_name}',
                    (x1, text_y),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.6,
                    color,
                    2,
                    cv2.LINE_AA,
                )
        return drawn

    def load_prediction_image(result, prediction_dir):
        saved_prediction_path = prediction_dir / Path(result.path).name
        if saved_prediction_path.exists():
            saved_prediction = cv2.imread(str(saved_prediction_path))
            if saved_prediction is not None:
                return saved_prediction

        plotted = result.plot()
        return cv2.cvtColor(plotted, cv2.COLOR_RGB2BGR)

    def save_side_by_side_comparison(image_path, label_path, result, prediction_dir, comparison_dir):
        original_bgr = cv2.imread(str(image_path))
        if original_bgr is None:
            return None

        ground_truth_bgr = draw_labeled_boxes(original_bgr, label_path, TEST_CLASS_NAMES)
        prediction_bgr = load_prediction_image(result, prediction_dir)
        if prediction_bgr is None:
            return None

        if ground_truth_bgr.shape[:2] != prediction_bgr.shape[:2]:
            prediction_bgr = cv2.resize(
                prediction_bgr,
                (ground_truth_bgr.shape[1], ground_truth_bgr.shape[0]),
                interpolation=cv2.INTER_LINEAR,
            )

        header_height = 40
        image_height, image_width = ground_truth_bgr.shape[:2]
        canvas = 255 * np.ones((image_height + header_height, image_width * 2, 3), dtype=np.uint8)
        canvas[header_height:, :image_width] = ground_truth_bgr
        canvas[header_height:, image_width:] = prediction_bgr

        cv2.putText(canvas, 'Ground Truth', (20, 28), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 0), 2, cv2.LINE_AA)
        cv2.putText(canvas, 'Prediction', (image_width + 20, 28), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 0), 2, cv2.LINE_AA)
        cv2.line(canvas, (image_width, 0), (image_width, image_height + header_height), (180, 180, 180), 2)

        comparison_path = comparison_dir / Path(image_path).name
        cv2.imwrite(str(comparison_path), canvas)
        return comparison_path

    if TEST_HAS_LABELS:
        TEST_YAML_PATH = TEST_RUNS_DIR / 'test_data.yaml'
        external_test_config = {
            'path': str(PROJECT_DIR),
            # Ultralytics still requires both train and val keys in detection YAMLs,
            # even when we only run model.val(..., split='test').
            'train': str(TEST_IMAGE_DIR),
            'val': str(TEST_IMAGE_DIR),
            'test': str(TEST_IMAGE_DIR),
            'nc': len(TEST_CLASS_NAMES),
            'names': TEST_CLASS_NAMES,
        }
        with open(TEST_YAML_PATH, 'w') as f:
            yaml.safe_dump(external_test_config, f, sort_keys=False)
        print('Test YAML:', TEST_YAML_PATH)
    else:
        print('No ground-truth test labels detected. Metrics plots will be skipped, but prediction outputs and count CSVs will still be saved.')

    for run in TEST_RUNS:
        print('=' * 80)
        print(f"Running test for {run['run_name']}")
        staged_model_path = stage_model_for_inference(run['best_model'], run['run_name'])
        model = YOLO(str(staged_model_path))

        run_root_dir = TEST_RUNS_DIR / run['run_name']
        prediction_dir = run_root_dir / 'predictions'
        model_report_dir = run_root_dir / 'reports'
        comparison_dir = run_root_dir / 'comparisons'
        run_root_dir.mkdir(parents=True, exist_ok=True)
        prediction_dir.mkdir(parents=True, exist_ok=True)
        model_report_dir.mkdir(parents=True, exist_ok=True)
        if TEST_HAS_LABELS and TEST_SAVE_COMPARISON_IMAGES:
            comparison_dir.mkdir(parents=True, exist_ok=True)

        run_per_class_rows = []

        summary_row = {
            'run_name': run['run_name'],
            'dataset': run.get('dataset'),
            'yolo_size': run.get('yolo_size'),
            'epochs': run.get('epochs'),
            'model_path': str(run['best_model']),
            'staged_model_path': str(staged_model_path),
            'test_image_dir': str(TEST_IMAGE_DIR),
            'test_label_dir': str(TEST_LABEL_DIR),
            'test_images': len(test_image_paths),
            'test_label_files': len(test_label_paths),
            'run_root_dir': str(run_root_dir),
            'prediction_dir': str(prediction_dir),
            'reports_dir': str(model_report_dir),
            'comparison_dir': str(comparison_dir) if TEST_HAS_LABELS and TEST_SAVE_COMPARISON_IMAGES else '',
        }

        if TEST_HAS_LABELS:
            validation_metrics = model.val(
                data=str(TEST_YAML_PATH),
                split='test',
                imgsz=TEST_IMAGE_SIZE,
                device=TEST_DEVICE,
                project=str(run_root_dir),
                name='reports',
                exist_ok=True,
                plots=True,
            )

            per_class_ap50 = list(getattr(validation_metrics.box, 'ap50', []))
            per_class_map = list(getattr(validation_metrics.box, 'maps', []))

            summary_row.update({
                'test_map50_95': float(validation_metrics.box.map),
                'test_map50': float(validation_metrics.box.map50),
                'test_precision': float(validation_metrics.box.mp),
                'test_recall': float(validation_metrics.box.mr),
            })

            for class_index, class_name in enumerate(TEST_CLASS_NAMES):
                ap50_value = float(per_class_ap50[class_index]) if class_index < len(per_class_ap50) else None
                map_value = float(per_class_map[class_index]) if class_index < len(per_class_map) else None
                per_class_row = {
                    'run_name': run['run_name'],
                    'dataset': run.get('dataset'),
                    'yolo_size': run.get('yolo_size'),
                    'epochs': run.get('epochs'),
                    'class_name': class_name,
                    'test_ap50': ap50_value,
                    'test_map50_95': map_value,
                }
                run_per_class_rows.append(per_class_row)
                external_test_per_class_rows.append(per_class_row)
                summary_row[f'test_ap50_{class_name}'] = ap50_value
                summary_row[f'test_map50_95_{class_name}'] = map_value

        results = model.predict(
            source=str(TEST_IMAGE_DIR),
            imgsz=TEST_IMAGE_SIZE,
            conf=TEST_CONF,
            device=TEST_DEVICE,
            project=str(run_root_dir),
            name='predictions',
            exist_ok=True,
            save=TEST_SAVE_PREDICTION_IMAGES,
            save_txt=TEST_SAVE_PREDICTION_LABELS,
            save_conf=TEST_SAVE_PREDICTION_CONF,
        )

        predicted_class_counts = {class_name: 0 for class_name in TEST_CLASS_NAMES}
        total_predictions = 0
        comparison_images_saved = 0

        for result in results:
            per_image_counts = {class_name: 0 for class_name in TEST_CLASS_NAMES}
            class_ids = []
            if result.boxes is not None and result.boxes.cls is not None:
                class_ids = result.boxes.cls.int().tolist()

            total_predictions += len(class_ids)
            for class_id in class_ids:
                if 0 <= class_id < len(TEST_CLASS_NAMES):
                    class_name = TEST_CLASS_NAMES[class_id]
                    per_image_counts[class_name] += 1
                    predicted_class_counts[class_name] += 1

            image_path = Path(result.path)
            image_row = {
                'run_name': run['run_name'],
                'dataset': run.get('dataset'),
                'yolo_size': run.get('yolo_size'),
                'epochs': run.get('epochs'),
                'image_name': image_path.name,
                'num_predictions': len(class_ids),
            }
            image_row.update({f'pred_{class_name}': per_image_counts[class_name] for class_name in TEST_CLASS_NAMES})

            if TEST_HAS_LABELS and TEST_SAVE_COMPARISON_IMAGES:
                comparison_path = save_side_by_side_comparison(
                    image_path=image_path,
                    label_path=test_label_map.get(image_path.stem),
                    result=result,
                    prediction_dir=prediction_dir,
                    comparison_dir=comparison_dir,
                )
                image_row['comparison_image'] = str(comparison_path) if comparison_path else ''
                if comparison_path:
                    comparison_images_saved += 1

            external_test_image_rows.append(image_row)

        summary_row['pred_total'] = total_predictions
        summary_row['comparison_images_saved'] = comparison_images_saved
        for class_name in TEST_CLASS_NAMES:
            summary_row[f'pred_{class_name}'] = predicted_class_counts[class_name]

        run_per_class_df = pd.DataFrame(run_per_class_rows)
        run_per_class_path = model_report_dir / 'test_per_class.csv'

        if not run_per_class_df.empty:
            run_per_class_df.to_csv(run_per_class_path, index=False)

        summary_row['model_per_class_csv'] = str(run_per_class_path) if not run_per_class_df.empty else ''

        external_test_summary_rows.append(summary_row)
        print(f'Saved predictions for {len(results)} images to {prediction_dir}.')
        if TEST_HAS_LABELS and TEST_SAVE_COMPARISON_IMAGES:
            print(f'Saved {comparison_images_saved} comparison images to {comparison_dir}.')
        print('Model report directory:', model_report_dir)
        del model
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    new_external_test_summary_df = pd.DataFrame(external_test_summary_rows)
    external_test_per_class_df = pd.DataFrame(external_test_per_class_rows)
    external_test_image_df = pd.DataFrame(external_test_image_rows)

    test_summary_path = TEST_RUNS_DIR / 'test_summary.csv'
    if test_summary_path.exists():
        test_summary_df = pd.read_csv(test_summary_path)
        test_summary_df = test_summary_df[~test_summary_df['run_name'].isin(new_external_test_summary_df['run_name'])]
        test_summary_df = pd.concat([test_summary_df, new_external_test_summary_df], ignore_index=True)
    else:
        test_summary_df = new_external_test_summary_df

    if TEST_HAS_LABELS and not test_summary_df.empty:
        test_summary_df = test_summary_df.sort_values(
            ['test_map50_95', 'test_map50', 'test_precision', 'test_recall', 'run_name'],
            ascending=[False, False, False, False, True],
        ).reset_index(drop=True)
        if 'test_rank' in test_summary_df.columns:
            test_summary_df = test_summary_df.drop(columns=['test_rank'])
        test_summary_df.insert(0, 'test_rank', range(1, len(test_summary_df) + 1))
    else:
        if 'test_rank' in test_summary_df.columns:
            test_summary_df = test_summary_df.drop(columns=['test_rank'])
        test_summary_df = test_summary_df.sort_values(
            ['yolo_size', 'dataset', 'epochs', 'run_name'],
            na_position='last',
        ).reset_index(drop=True)

    test_summary_df.to_csv(test_summary_path, index=False)

    print('Test summary CSV:', test_summary_path)

    display(new_external_test_summary_df)
    if not external_test_per_class_df.empty:
        display(external_test_per_class_df)
    display(external_test_image_df.head(20))


## 7. Save the Trained Models for the Dashboard

Each dataset/model-size combination exports to a separate file so none of the eight runs overwrite each other.

In [ ]:
# Code Cell 10
EXPORT_DIR.mkdir(parents=True, exist_ok=True)
export_rows = []

for run in RUNS:
    dashboard_model = EXPORT_DIR / f"{run['run_name']}_best.pt"
    shutil.copy2(run['best_model'], dashboard_model)
    export_rows.append({
        'dataset': run['dataset'],
        'yolo_size': run['yolo_size'],
        'epochs': run['epochs'],
        'run_name': run['run_name'],
        'exported_model': dashboard_model,
        'local_models_path': f"models/{run['run_name']}.pt",
    })

export_table = pd.DataFrame([{k: str(v) if isinstance(v, Path) else v for k, v in row.items()} for row in export_rows])
display(export_table)
print('Export directory:', EXPORT_DIR)